# Lecture: Continuous Batching and PagedAttention (vLLM ideas)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain static batching's two taxes (wait for the bus to fill; pad short rides to the longest) in one sentence each (§4).
- Define TTFT vs TPOT and say which optimization moves which (§5).
- Explain PagedAttention block tables and prefix sharing with the kB saved (§6–§7).
- Read a scheduler comparison and argue when the continuous win is biggest (§4 + §8).


## The big idea

Static batching is a school bus: it waits until 4 kids arrive, then drives every kid to the *farthest* stop — short rides get padded with empty sitting (**pad waste**), and early kids wait (**high TTFT**). **Continuous batching** is a taxi line: the moment one rider gets out, the next hops in at that step (**iteration-level scheduling**) — nobody waits for the slowest trip, no seat rides empty, makespan drops ~1.33x. **PagedAttention** fixes the parking lot: instead of reserving every car the worst-case garage (**contiguous max**, 1511.4 kB), it hands out small **pages** (blocks of 16 tokens, 995.3 kB) and only as many as each trip needs — plus carpools sharing the first 32 tokens (**prefix sharing**) pay once. Analogy: static = fixed banquet seating for the longest speech; continuous + paged = a busy café that reseats each freed chair and splits checks per page.


In [ ]:
import numpy as np  # bring in numpy for the toy wait times
import matplotlib.pyplot as plt  # bring in pyplot for the wait bars
import random  # bring in stdlib random for the seed habit
random.seed(18)  # fix stdlib seed so jitter matches each run
np.random.seed(18)  # fix numpy seed so the lecture matches each run
static = np.array([89.7, 89.7, 95.0, 110.0])  # toy TTFT ms: all wait for fill + drain
cont = np.array([8.0, 25.0, 40.0, 55.0])  # toy TTFT ms: early riders leave early
print('static TTFT:', static.tolist())  # show everyone waits for the slowest
print('continuous TTFT:', cont.tolist())  # show staggered quick starts
print('median static:', round(float(np.median(static)), 1))  # print 92.3 ms typical wait
print('median continuous:', round(float(np.median(cont)), 1))  # print 32.5 ms typical wait
y = np.arange(4)  # four toy riders in arrival order
h = 0.35  # bar half-height so paired bars sit side by side
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the pairs
ax.barh(y + h / 2, static, h, label='static')  # long bars are bus waits
ax.barh(y - h / 2, cont, h, label='continuous')  # short bars are taxi waits
ax.set_yticks(y, ['r1', 'r2', 'r3', 'r4'])  # name each rider on the axis
ax.set_xlabel('TTFT ms (shorter starts faster)')  # label defines wait direction
ax.set_title('Taxis start faster than the school bus')  # title in plain words
ax.legend()  # key telling which color is which scheduler
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the wait bars
print('Lesson: the win is waiting, not word speed.')  # one-sentence takeaway
print('RECALIBRATE: constants above are the author CPU run (SEED=18); paste your results/sched.csv + sweeps.csv row to replace them - ordering/shape is the claim, digits are not')


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 removes waiting, §5 clocks it, §6–§7 un-waste memory.

| Term | One-liner | Section |
|---|---|---|
| Static batching | Wait for B requests, run all to the longest length; pays waiting + padding taxes | §4 |
| Continuous batching | Recheck every step: admit arrivals now, backfill freed seats; zero padding | §4 |
| TTFT | Time to first token: queue wait + prefill until the first word | §5 |
| TPOT | Time per output token: steady decode gap (barely moves here) | §5 |
| Makespan / throughput | Wall time to drain all N requests / tokens per second overall | §5 |
| PagedAttention | Store each request's keys/values in small blocks (pages), not one max slab | §6 |
| Prefix sharing | Requests with the same first 32 tokens share those pages once | §7 |


### §4 Static vs continuous — the two taxes, removed

**In one sentence: static batching taxes every request twice — waiting until the batch fills (head-of-line blocking) and padding every member to the longest output — while continuous batching admits on arrival and frees each seat the moment its reply ends.**

Tax 1 (waiting): the bus departs only when all 4 kids arrive, so rider 1's TTFT includes riders 2–4's lateness. Tax 2 (padding): with outputs [61, 12, 43, 20], everyone rides 61 steps — the 12-token request burns 49 fake steps, averaging 30.5 wasted tokens per request. Continuous pays neither: median TTFT 89.7 → 42.2 ms, makespan 1.20 → 0.90 s (×1.33), pad waste 30.5 → 0.0. Watch it: the TTFT bars plus the makespan math, all printed.


In [ ]:
import numpy as np  # reuse numpy for the policy arrays
import matplotlib.pyplot as plt  # reuse pyplot for the TTFT bars
np.random.seed(18)  # fix the seed for reproducibility
ttft_static, ttft_cont = 89.7, 42.2  # median first-word waits per policy (ms)
span_static, span_cont = 1.20, 0.90  # makespan to drain 48 requests per policy (s)
waste_static, waste_cont = 30.5, 0.0  # mean padding tax per request (tokens)
print('TTFT med ms: static', ttft_static, 'vs continuous', ttft_cont)  # print 89.7 vs 42.2: waiting halved
print('makespan s:', span_static, 'vs', span_cont)  # print 1.20 vs 0.90: drain 1.33x faster
print('speedup:', round(span_static / span_cont, 2))  # print 1.33: the headline win
print('pad waste toks/req:', waste_static, 'vs', waste_cont)  # print 30.5 vs 0.0: padding gone
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the TTFT bars
ax.bar(['static TTFT', 'continuous TTFT'], [ttft_static, ttft_cont], color=['tab:red', 'tab:green'])  # waiting halved
ax.set_ylabel('median TTFT ms (lower starts faster)')  # label defines the wait direction
ax.set_title('Waiting halved: 89.7 -> 42.2 ms (makespan x1.33)')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

### §5 TTFT, TPOT, makespan — three clocks, three owners

**In one sentence: TTFT (queue wait + prefill) belongs to the scheduler, TPOT (steady seconds per decoded word) belongs to the hardware batch width, and makespan (wall time for all N) is the bill both optimizations split.**

Dissect one request: it queues 60.0 ms behind earlier buses, prefills 29.7 ms (prompt read + first word) → TTFT 89.7 ms; then decodes 8 words at TPOT 0.47 ms → 3.76 ms more, end-to-end 93.5 ms. Continuous batching crushes the 60.0 ms queue slice and shares prefill — but TPOT barely moves, because word speed was never the bottleneck. Throughput still rises ×1.33: same words, less waiting, zero padding. Watch it: the annotated timeline with every segment printed.


In [ ]:
import numpy as np  # reuse numpy for the segment arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the annotated timeline
np.random.seed(18)  # fix the seed for reproducibility
queue, prefill = 60.0, 29.7  # ms waiting for the bus + ms reading the prompt
ttft = round(queue + prefill, 1)  # first-word wait: 89.7 ms
tpot, n = 0.47, 8  # steady gap per word + words decoded
dec = round(tpot * n, 2)  # decode slice: 3.76 ms
e2e = round(ttft + dec, 1)  # end-to-end for this request: 93.5 ms
print('queue', queue, '+ prefill', prefill, '= TTFT', ttft)  # print the scheduler-owned slice
print('TPOT', tpot, 'x', n, '= decode', dec)  # print the hardware-owned slice
print('end-to-end:', e2e)  # print 93.5: the request's whole bill
fig, ax = plt.subplots(figsize=(8, 3))  # wide flat figure for the timeline
ax.barh([0], [queue], color='tab:red', label='queue 60.0')  # red: waiting (scheduler tax)
ax.barh([0], [prefill], left=[queue], color='tab:orange', label='prefill 29.7')  # orange: prompt read
ax.barh([0], [dec], left=[ttft], color='tab:green', label='decode 3.76')  # green: steady words
ax.set_xlim(0, 100)  # fix range to fit the whole 93.5 ms bill
ax.set_yticks([])  # hide y: time is the only axis
ax.set_xlabel('ms since arrival (one request)')  # label the time axis
ax.set_title('One request: TTFT 89.7 (wait+prefill) + decode 3.76 = 93.5')  # title in printed numbers
ax.legend(fontsize=8, loc='lower right')  # show which color is which slice
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the timeline

### §6 PagedAttention — pay per page, not per worst case

**In one sentence: PagedAttention stores each request's remembered keys/values in fixed blocks (pages of 16 tokens) tracked by a block table, instead of reserving the worst-case length for every request — 1511.4 → 995.3 kB here (1.52x).**

The **KV cache** (saved keys/values so old tokens are never recomputed) must live somewhere. Contiguous reservation books every request the longest trip's garage: 48 requests × worst length × fp16 bytes = 1511.4 kB, mostly empty. Paging books only the blocks each trip actually touches: 995.3 kB, freeing 516.1 kB with zero math change — same numbers, smarter parking. Watch it: the block-table grid (each request owns only its shaded pages) plus the memory bars.


In [ ]:
import numpy as np  # reuse numpy for the memory arrays
import matplotlib.pyplot as plt  # reuse pyplot for grid + bars
np.random.seed(18)  # fix the seed for reproducibility
contig, paged = 1511.4, 995.3  # KV kB: worst-case reservation vs pay-per-page
saved = round(contig - paged, 1)  # freed memory with identical math
print('contiguous kB:', contig)  # print 1511.4: every garage max-sized
print('paged-b16 kB:', paged)  # print 995.3: only touched pages booked
print('saved kB:', saved)  # print 516.1: freed for more requests
print('ratio:', round(contig / paged, 2))  # print 1.52: the paging win
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: block grid + memory
ax[0].set_xlim(0, 7)  # fix grid range: 6 page columns
ax[0].set_ylim(0, 5)  # fix grid range: 4 request rows
ax[0].axis('off')  # hide axes: this is a diagram
lens = [5, 2, 4, 3]  # pages actually touched by requests r1..r4 (of 6 reserved)
for r, L in enumerate(lens):  # draw one row per request
    for b in range(6):  # draw one page slot per column
        col = 'lightgreen' if b < L else 'whitesmoke'  # green = booked, gray = never reserved
        ax[0].add_patch(plt.Rectangle((b + 0.1, r + 0.3), 0.8, 0.6, facecolor=col, edgecolor='k'))  # page slot
    ax[0].text(6.5, r + 0.6, f'r{r + 1}:{L}pg', va='center', fontsize=8)  # label pages owned
ax[0].set_title('Block table: own only touched pages')  # left title
ax[1].bar(['contiguous', 'paged-b16'], [contig, paged], color=['tab:red', 'tab:green'])  # memory bars
ax[1].set_ylabel('KV kB fp16 (lower is smaller)')  # label defines memory direction
ax[1].set_title('Paging saves 1.52x (516.1 kB freed)')  # title in printed numbers
plt.tight_layout()  # stop titles and labels from overlapping
plt.show()  # display the figure

### §7 Prefix sharing — carpool the common road

**In one sentence: when G requests start with the same S tokens (system prompt, few-shot examples), prefix sharing stores those pages once and points every request at the shared copy — 4 requests × a 32-token prefix (2 blocks each) drops from 8 blocks to 2, saving 24.6 kB.**

Same opening paragraph, four riders: without sharing each pays 2 blocks (8 total); with sharing, one 2-block copy serves all four (copy-on-write if anyone diverges later). At 4.096 kB per block (K+V × 2 layers × width 64 × 16 tokens, fp16), 6 freed blocks = 24.6 kB — on top of paging's win, for free. Watch it: the shared-blocks diagram with the savings printed.


In [ ]:
import numpy as np  # reuse numpy for the block arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the sharing diagram
np.random.seed(18)  # fix the seed for reproducibility
G, S, BLK = 4, 32, 16  # grouped requests, shared tokens, block size
shared = 2  # 32 shared tokens = 2 blocks stored once
separate = G * 2  # without sharing: each of 4 pays 2 blocks
blk_kb = 2 * 2 * 64 * 16 / 1e3  # kB per block: K+V x 2 layers x d64 x 16 toks fp16
save = round((separate - shared) * blk_kb, 1)  # kB freed by sharing
print('blocks separate:', separate, 'vs shared:', shared)  # print 8 vs 2: one copy serves four
print('kB per block:', round(blk_kb, 3))  # print 4.096: the page price
print('prefix saves kB:', save)  # print 24.6: carpool discount
fig, ax = plt.subplots(figsize=(8, 3.5))  # wide figure for the sharing diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
ax.text(1.5, 2.0, 'prefix\n32 toks\n(2 blocks)', ha='center', va='center', fontsize=9,  # left: shared road
        bbox=dict(boxstyle='round', facecolor='lightgreen'))  # green = stored once
for i, y in enumerate([3.0, 2.4, 1.6, 1.0]):  # draw one rider row each
    ax.text(5, y, f'req{i + 1} rest ->', ha='center', va='center', fontsize=8)  # rider's private tail
    ax.annotate('', xy=(2.8, 2.0), xytext=(4.2, y), arrowprops=dict(arrowstyle='->', color='gray'))  # shared-copy pointer
ax.text(8, 2.0, '8 blocks\n-> 2 blocks\nsave 24.6 kB', ha='center', va='center', fontsize=9,  # right: savings
        bbox=dict(boxstyle='round', facecolor='wheat'))  # wheat = discount
ax.set_title('Prefix sharing: 4 riders, one copy of the first 32 tokens')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

## §8 How it all fits together — the system map

One serving run flows through every § term in order. The **arrival stream** (48 requests, skewed lengths, bursty times) hits the **scheduler**: static batches wait-and-pad (§4's taxes) while continuous admits-and-backfills each step. Running requests live in **paged KV** (block tables, §6; shared prefixes, §7) instead of max-sized slabs. Out come **completions** stamped with **TTFT/TPOT** (§5) and the makespan/throughput bill. The map below pins each term to its station; the trace stamps 4 requests with their TTFT/TPOT numbers.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import random  # bring in stdlib random only to satisfy the seed habit (unused here)
random.seed(18)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(11, 3.5))  # wide figure for the serving pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 15)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('arrivals\n48 skewed', 1), ('scheduler\n§4 cont.', 4.2), ('paged KV\n§6+§7', 7.4), ('TTFT/TPOT\n§5 stamp', 10.2), ('drained\nx1.33', 13)]  # five pipeline stations
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = pipeline
for xa, xb in [(1.9, 3.2), (5.1, 6.4), (8.3, 9.3), (11.2, 12.3)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2),  # draw a forward arrow left -> right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.text(7, 3.4, 'FLOW: stream -> schedule per step -> page memory (§4, §6-§7)', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(7, 0.9, 'BILL: TTFT 42.2 ms, waste 0, makespan x1.33 (§4-§5)', ha='center', fontsize=9, fontweight='bold', color='darkred')  # bill banner
ax.set_title('System map: arrivals -> scheduler -> paged KV -> completions')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the tiny medians
np.random.seed(18)  # fix the seed for reproducibility
ttft = [8.0, 25.0, 40.0, 55.0]  # per-request first-word waits, continuous policy (ms)
tpot = [0.47, 0.45, 0.48, 0.49]  # per-request steady word gaps (ms)
outs = [61, 12, 43, 20]  # words each request asked for (skewed: the static tax base)
for i, (a, b, o) in enumerate(zip(ttft, tpot, outs)):  # stamp each request
    print(f'r{i + 1}: out {o} toks | TTFT {a} ms | TPOT {b} ms')  # one stamped line per request
print('median TTFT:', round(float(np.median(ttft)), 1))  # print 32.5: typical start
print('static paid: all TTFT >= 89.7 + 30.5 pad toks/req')  # print the removed taxes (§4)

## Bridge to the lab

Open `notebook.ipynb` now. Its cells simulate exactly the pipeline above (event-driven sims plus one real forward to anchor costs — no cluster needed):

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `SEED = 18` habit as this lecture.
2. **Cell 2 (latency model + workload)** — times a real tiny forward for prefill/decode costs, then samples 48 skewed requests over a 0.15 s arrival window — our trace's skew by hand.
3. **Cell 3 (schedulers)** — static waits-for-B and runs to max-len (§4's taxes) vs continuous prefills-on-arrival and backfills each step — TTFT 89.7 → 42.2, makespan ×1.33.
4. **Cell 4 (metrics + paging)** — TTFT/TPOT/makespan/throughput into `results/sched.csv` (§5), block tables vs contiguous plus prefix sharing into `results/blocks.csv` (§6–§7: 1511.4 → 995.3 kB, 24.6 saved).
5. **Cell 5 (sweeps + figure)** — burst windows and block sizes (small blocks waste less) into `results/sweeps.csv` and `results/sched.png`.
6. **Cell 6 (cleanup)** — states the model limits honestly (joint-batch decode cost, 1-layer proxy): direction transfers, exact ratios need a GPU rerun.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Continuous batching barely moves TPOT but halves TTFT. Why still worth it?**

*Answer: the bottleneck was waiting, not word speed (§4–§5). Static queues arrivals and drains to max-len, so TTFT dominates; removing waiting halves starts (89.7 → 42.2) and lifts throughput ×1.33 at near-equal word speed. Trace r1–r4 in the §8 numbers.*

2. **Why does static waste 30.5 tokens/request while continuous wastes 0?**

*Answer: static pads every member to the longest output (§4: out_max 61 in the trace), burning fake steps; continuous frees each seat the moment its own reply ends, so no padding exists by construction.*

3. **PagedAttention saves 1.52x (516.1 kB) with zero math change, plus 24.6 kB from prefixes. How?**

*Answer: paging (§6) books only touched 16-token blocks instead of reserving every request's worst-case slab, and prefix sharing (§7) stores the common 32 tokens once for 4 riders (8 blocks → 2). Same numbers, smarter parking plus carpooling.*
